# VadCLIP V7 — Suy luận với cửa sổ chồng lấn

Giả thuyết là snippet sát mép block thiếu ngữ cảnh, làm điểm frame kém ổn định.
So cùng checkpoint V0, feature và evaluator trên UCF-Crime và XD-Violence.
V0 giữ block gốc. V7 dịch cửa sổ một nửa block mỗi bước và lấy logit từ cửa sổ
có khoảng cách nhỏ nhất tới hai mép nội bộ lớn nhất. Mép thật của video không bị
tính là mép nhân tạo. Hòa điểm chọn cửa sổ bắt đầu sớm hơn. Giữ thứ tự và số
snippet trước khi áp cùng công thức điểm của V0. Video vừa trong một block đi
qua forward gốc. Thiết kế offline, dùng cả ngữ cảnh trước và sau snippet.

## Add Input trên Kaggle

| Input | File trong Dataset |
|---|---|
| UCF-Crime test 290 | ucfclipfeatures_test290.zip |
| UCF-Crime checkpoint công bố | vadclip_ucf_checkpoint_upload.zip hoặc model_ucf.pth |
| XD-Violence test 800 bản Kaggle V2 | vadclip_xd_test800_kaggle_v2.zip |
| XD-Violence checkpoint công bố | vadclip_xd_checkpoint.zip hoặc model_xd.pth |

Gắn bốn gói từng dùng ở V0/V6. Không cần gói train hoặc checkpoint V6.
Nếu không thấy checkpoint UCF, gắn Dataset checkpoint riêng trong Add Input.
Nếu tên file khác, dùng Copy path trên Kaggle rồi đặt UCF_CHECKPOINT trong ô cấu hình.
Nếu smoke XD đã đạt, đặt DATASETS = ("ucf",) cho lượt smoke kiểm lại UCF.

Bật GPU T4 và Internet để lấy mã, nhãn chính thức và OpenAI CLIP ViT-B/16.
Chạy RUN_MODE = "smoke" trước, rồi RUN_MODE = "full". DATASETS mặc định chấm cả hai.
Smoke chạy forward trên mẫu, gồm video dài hơn block. Thời gian và VRAM
được ghi để ước lượng chi phí lượt full. Lượt full ghi comparison.csv,
summary.json, run.json và predictions.npz vào /kaggle/working/vadclip_v7/run_*/.

Đọc chênh V7 − V0 trên C-Branch UCF (ROC-AUC, Ano-AUC, AP) và A-Branch XD
(frame AP, ROC-AUC). Chẩn đoán tách vùng cách mép block cũ tối đa 8 snippet
và các đoạn dương dài tối đa 48 frame. Các số cố định trước khi chạy, không dùng
kết quả test để chọn lại. Vì official test đã được xem trước, đây là phép thử thăm dò.


## Đo latency inference

- Đổi `RUN_MODE = "full"` để thống kê đủ 290 video UCF và 800 video XD.
- `LATENCY_WARMUP = 3`: chạy thêm 3 lần trên video đầu mỗi dataset/cấu hình, không tính vào thống kê. Hình dạng mới vẫn có thể có chi phí khởi tạo.
- `LATENCY_REPEATS = 3`: đo 3 lần/video; có thể tăng lên 5. Thời gian inference tăng tương ứng, thời gian train V20 được ghi riêng như trước.
- Đồng bộ CUDA trước/sau mỗi lần; đo wall time từ feature/mask đã trên GPU đến logits. Bao gồm chia/ghép cửa sổ V7 và head/cân bằng V20. Không gồm đọc file, padding CPU, truyền CPU–GPU, sigmoid/softmax, giải mã video, trích xuất feature CLIP, tải model, train hoặc tính metric. Đây là inference offline trên feature, chưa phải latency toàn hệ thống hay độ trễ cảnh báo streaming.
- Trả lại logits của lần đo đầu để evaluator giữ nguyên cách tính điểm. Warm-up bảo toàn RNG. Không thay checkpoint, precision, split, seed hay evaluator.
- Mỗi video lấy median của các lần đo; mean/median/P95 là phân phối các median video. `ms_per_snippet` = tổng median video / tổng snippet hợp lệ, tính cả chi phí cửa sổ trùng. Không gọi snippet/s là FPS video thô.
- File mỗi cấu hình: `latency_per_video.csv` (từng lần đo), `latency_summary.json`, trường `latency` trong `run.json`. File gộp: `latency_comparison.csv`. Smoke chỉ là kiểm đường chạy, không dùng kết luận hiệu năng.
- So trên cùng loại GPU, input, RUN_MODE, warm-up/repeats; không chạy notebook khác đồng thời. Mean ms/video phụ thuộc độ dài video, vì vậy báo riêng từng dataset và kèm ms/snippet. Các trường `elapsed_seconds`/`inference_seconds` cũ là thời gian luồng thực thi, không dùng thay latency mới.


## Kết quả và log

Sau mỗi cấu hình, cell chạy in cả hai nhánh AUC/AP/Ano-AUC khi có, chỉ rõ nhánh chính,
chênh với mốc V0 đã tái lập, latency mean/median/P95/ms-snippet/snippet-s,
peak allocated VRAM và số tham số model/head. Cuối lượt in bảng tổng hợp và chênh
latency so V0 trong cùng lượt. Các bảng lưu trong benchmark_summary.csv/json;
cấu hình chính xác từng subprocess và SHA-256 runner lưu tại runtime_scripts/*.config.json.
Tất cả được thêm vào ZIP như các kết quả/cấu hình cũ. Số tham số là model nạp vào bộ nhớ,
bao gồm encoder đóng băng, không phải chỉ phần được thực thi khi dùng feature có sẵn.
Không tính FLOPs, Base/Novel, online delay hoặc end-to-end FPS trong giao thức này.
Smoke hiển thị benchmark N/A; latency smoke không dùng để kết luận benchmark full.


In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_MODE = "smoke"
DATASETS = ("ucf", "xd")
UCF_DATA_ROOT = None
UCF_CHECKPOINT = None
XD_DATA_ROOT = None
XD_FEATURE_ZIP = None
XD_CHECKPOINT = None
CLIP_WEIGHTS = None

LATENCY_WARMUP = 3
LATENCY_REPEATS = 3


## Chia cửa sổ và ghép logit


In [ ]:
WINDOW_SOURCE = r"""
import numpy as np
import torch


def window_plan(steps, width):
    # Map each valid snippet to exactly one overlapping window.
    if width < 2 or steps <= width:
        raise ValueError("Window plan requires a multi-block video.")
    stride = width // 2
    starts = list(range(0, steps - width, stride))
    starts.append(starts[-1] + stride)
    owner = np.full(steps, -1, dtype=np.int64)
    quality = np.full(steps, -1, dtype=np.int64)
    for index, start in enumerate(starts):
        stop = min(start + width, steps)
        local = np.arange(stop - start)
        left = local if start else width
        right = stop - start - 1 - local if stop < steps else width
        context = np.minimum(left, right)
        better = context > quality[start:stop]
        quality[start:stop][better] = context[better]
        owner[start:stop][better] = index
    if (owner < 0).any():
        raise RuntimeError("Window plan missed valid snippets.")
    return [(start, np.flatnonzero(owner[start:min(start + width, steps)] == i))
            for i, start in enumerate(starts)]


def overlap_forward(forward, images, mask, text, lengths, make_mask):
    # The upstream loader passes one video as [blocks, snippets, channels].
    sizes = torch.as_tensor(lengths).detach().cpu().reshape(-1)
    blocks, width, channels = images.shape
    if sizes.numel() != blocks or (sizes < 0).any() or (sizes > width).any():
        raise ValueError("Unexpected VadCLIP block lengths.")
    steps = int(sizes.sum())
    if steps <= width:
        return forward(images, mask, text, lengths)
    expected = (steps - torch.arange(blocks) * width).clamp(min=0, max=width)
    if not torch.equal(sizes, expected):
        raise ValueError("VadCLIP video blocks are not a contiguous padded sequence.")

    flat = images.reshape(-1, channels)[:steps]
    plan = window_plan(steps, width)
    for offset in range(0, len(plan), 8):
        chunk = plan[offset:offset + 8]
        batch = images.new_zeros((len(chunk), width, channels))
        valid_lengths = torch.tensor([min(width, steps - start) for start, _ in chunk])
        for row, ((start, _), length) in enumerate(zip(chunk, valid_lengths.tolist(), strict=True)):
            batch[row, :length] = flat[start:start + length]
        batch_mask = make_mask(valid_lengths, width).to(images.device)
        text_features, logits_c, logits_a = forward(batch, batch_mask, text, valid_lengths)
        if offset == 0:
            result_c = logits_c.new_zeros((blocks * width, *logits_c.shape[2:]))
            result_a = logits_a.new_zeros((blocks * width, *logits_a.shape[2:]))
        for row, (start, positions) in enumerate(chunk):
            indices = torch.as_tensor(positions, device=images.device)
            result_c[start + indices] = logits_c[row, indices]
            result_a[start + indices] = logits_a[row, indices]
    return (text_features, result_c.reshape(blocks, width, *result_c.shape[1:]),
            result_a.reshape(blocks, width, *result_a.shape[1:]))
"""

LATENCY_SOURCE = '"""Synchronized, serial, feature-to-logit latency (not raw-video latency)."""\n\nimport csv\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\nclass InferenceLatency:\n    def __init__(self, warmup, repeats):\n        if type(warmup) is not int or warmup < 0:\n            raise ValueError("LATENCY_WARMUP must be an integer >= 0")\n        if type(repeats) is not int or repeats < 1:\n            raise ValueError("LATENCY_REPEATS must be an integer >= 1")\n        self.warmup = warmup\n        self.repeats = repeats\n        self.rows = []\n\n    def measure(self, forward, device, video_id, steps, shape, dtype):\n        """Input tensors already on device; return the first measured logits."""\n        device = torch.device(device)\n        steps = int(steps)\n        if steps < 1:\n            raise ValueError("Latency requires at least one valid snippet")\n\n        def synchronize():\n            if device.type == "cuda":\n                torch.cuda.synchronize(device)\n\n        if not self.rows:\n            # Warm up once per variant/dataset process. Preserve the RNG state.\n            devices = []\n            if device.type == "cuda":\n                devices = [device.index if device.index is not None else torch.cuda.current_device()]\n            with torch.random.fork_rng(devices=devices):\n                with torch.no_grad():\n                    for _ in range(self.warmup):\n                        forward()\n            synchronize()\n        first = None\n        with torch.no_grad():\n            for repeat in range(self.repeats):\n                synchronize()\n                started = time.perf_counter()\n                result = forward()\n                synchronize()\n                milliseconds = (time.perf_counter() - started) * 1000\n                if not np.isfinite(milliseconds) or milliseconds <= 0:\n                    raise ValueError("Non-positive or non-finite inference duration")\n                if repeat == 0:\n                    first = result\n                self.rows.append({\n                    "video_id": str(video_id), "repeat": repeat,\n                    "snippets": steps, "input_shape": str(tuple(shape)), "input_dtype": str(dtype),\n                    "latency_ms": milliseconds, "ms_per_snippet": milliseconds / steps,\n                })\n                del result\n        return first\n\n    def save(self, output, dataset, variant, mode, device):\n        if not self.rows:\n            raise ValueError("No measured inference calls")\n        grouped = {}\n        for row in self.rows:\n            grouped.setdefault(row["video_id"], []).append(row)\n        if any(len(rows) != self.repeats for rows in grouped.values()):\n            raise ValueError("Expected exactly LATENCY_REPEATS measurements per video")\n        per_video = np.array([\n            np.median([row["latency_ms"] for row in rows]) for rows in grouped.values()\n        ])\n        snippets = sum(rows[0]["snippets"] for rows in grouped.values())\n        total_ms = float(per_video.sum())\n        summary = {\n            "dataset": dataset, "variant": variant, "mode": mode,\n            "scope": "device-resident padded features/mask -> logits, including window stitching "\n                     "and temporal head when enabled; synchronized wall clock; serial execution",\n            "excluded": "file I/O, CPU padding, H2D/D2H transfer, sigmoid/softmax scoring, "\n                        "video decode, CLIP visual feature extraction, model load, metrics, training",\n            "warmup_calls_first_video": self.warmup, "repeats_per_video": self.repeats,\n            "warmup_policy": "first input only; new shapes may still incur first-use overhead",\n            "video_count": len(grouped), "measurement_count": len(self.rows),\n            "snippets": snippets,\n            "aggregation": "median over repeats per video, then distribution across videos; "\n                           "ms/snippet uses sum of video medians / sum of valid snippets",\n            "mean_ms_per_video": float(per_video.mean()),\n            "median_ms_per_video": float(np.median(per_video)),\n            "p95_ms_per_video": float(np.percentile(per_video, 95)),\n            "min_ms_per_video": float(per_video.min()),\n            "max_ms_per_video": float(per_video.max()),\n            "sum_video_medians_seconds": total_ms / 1000,\n            "ms_per_snippet": total_ms / snippets,\n            "snippets_per_second": snippets * 1000 / total_ms,\n            "torch": torch.__version__, "cuda_runtime": torch.version.cuda,\n            "device": str(device),\n            "gpu": torch.cuda.get_device_name(device) if torch.device(device).type == "cuda"\n                   else None,\n            "cudnn_benchmark": torch.backends.cudnn.benchmark,\n            "cudnn_deterministic": torch.backends.cudnn.deterministic,\n            "cuda_matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,\n            "cpu_threads": torch.get_num_threads(),\n            "dtype_policy": "unchanged from evaluator; see recorded input dtype",\n            "input_dtypes": sorted({row.get("input_dtype", "float32") for row in self.rows}),\n        }\n        output = Path(output)\n        output.mkdir(parents=True, exist_ok=True)\n        with (output / "latency_per_video.csv").open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(self.rows[0]))\n            writer.writeheader()\n            writer.writerows(self.rows)\n        (output / "latency_summary.json").write_text(\n            json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"\n        )\n        print(f"Latency {dataset}/{variant}: mean={summary[\'mean_ms_per_video\']:.3f} ms/video, "\n              f"P95={summary[\'p95_ms_per_video\']:.3f} ms/video, "\n              f"{summary[\'ms_per_snippet\']:.5f} ms/snippet", flush=True)\n        return summary\n\n\ndef collect_latency(output):\n    """Combine completed evaluator receipts, including a partial experiment."""\n    output = Path(output)\n    summaries = [json.loads(path.read_text(encoding="utf-8"))\n                 for path in sorted(output.rglob("latency_summary.json"))]\n    if summaries:\n        with (output / "latency_comparison.csv").open("w", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(summaries[0]))\n            writer.writeheader()\n            writer.writerows(summaries)\n    return summaries\n'

REPORT_SOURCE = '"""Console and JSON/CSV reports for the existing closed-set WSVAD protocol."""\n\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nV0_REFERENCE = {\n    "ucf": {"roc_auc": 0.8801992605166816, "ap": 0.3355725526808031,\n            "ano_auc": 0.7023225660564671},\n    "xd": {"roc_auc": 0.946812650099563, "ap": 0.8450448809004643},\n}\n\n\ndef count_parameters(model, bundle=None):\n    """Count resident model parameters, not FLOPs or feature extraction operations."""\n    base = sum(parameter.numel() for parameter in model.parameters())\n    extra = sum(parameter.numel() for parameter in bundle[0].parameters()) if bundle else 0\n    return {"base_model": base, "additional_head": extra, "total_resident": base + extra,\n            "scope": "loaded model including frozen encoders; plus separate head when enabled"}\n\n\ndef format_number(value, digits=4):\n    return "N/A" if value is None else f"{value:.{digits}f}"\n\n\ndef render_table(headers, rows, numeric=()):\n    """Use fixed character widths for notebook monospace console output."""\n    headers = [str(value) for value in headers]\n    cells = [["N/A" if value is None else str(value) for value in row] for row in rows]\n    if any(len(row) != len(headers) for row in cells):\n        raise ValueError("Table row does not match the number of columns")\n    widths = [max(len(header), *(len(row[index]) for row in cells))\n              if cells else len(header) for index, header in enumerate(headers)]\n\n    def line(row, heading=False):\n        return "| " + " | ".join(\n            cell.rjust(widths[index]) if index in numeric and not heading\n            else cell.ljust(widths[index]) for index, cell in enumerate(row)\n        ) + " |"\n\n    separator = "+-" + "-+-".join("-" * width for width in widths) + "-+"\n    return "\\n".join([separator, line(headers, True), separator,\n                      *(line(row) for row in cells), separator])\n\n\ndef flatten_report(report, source):\n    latency = report.get("latency") or {}\n    dataset = latency.get("dataset")\n    if dataset not in V0_REFERENCE:\n        raise ValueError(f"Unknown dataset in evaluator report: {source}")\n    primary = "classification" if dataset == "ucf" else "branch_2"\n    values = (report.get("metrics") or {}).get(primary) or {}\n    auc = values.get("roc_auc")\n    ap = values.get("ap", values.get("frame_ap"))\n    ano = values.get("ano_auc")\n    params = report.get("parameters") or {}\n    return {\n        "dataset": dataset, "variant": latency.get("variant"), "mode": latency.get("mode"),\n        "primary_branch": primary, "video_count": latency.get("video_count"),\n        "roc_auc_pct": None if auc is None else 100 * auc,\n        "ap_pct": None if ap is None else 100 * ap,\n        "ano_auc_pct": None if ano is None else 100 * ano,\n        "auc_vs_reference_v0_pp": None if auc is None else\n            100 * (auc - V0_REFERENCE[dataset]["roc_auc"]),\n        "ap_vs_reference_v0_pp": None if ap is None else\n            100 * (ap - V0_REFERENCE[dataset]["ap"]),\n        "mean_ms_per_video": latency.get("mean_ms_per_video"),\n        "median_ms_per_video": latency.get("median_ms_per_video"),\n        "p95_ms_per_video": latency.get("p95_ms_per_video"),\n        "ms_per_snippet": latency.get("ms_per_snippet"),\n        "snippets_per_second": latency.get("snippets_per_second"),\n        "peak_torch_vram_mib": report.get("peak_torch_vram_mib"),\n        "model_parameters": params.get("base_model"),\n        "additional_head_parameters": params.get("additional_head"),\n        "total_resident_parameters": params.get("total_resident"),\n        "gpu": latency.get("gpu"), "input_dtypes": ", ".join(latency.get("input_dtypes", [])),\n        "warmup": latency.get("warmup_calls_first_video"),\n        "repeats": latency.get("repeats_per_video"),\n        "visual_block_length": report.get("visual_block_length"),\n        "frames_per_feature": report.get("frames_per_feature"),\n        "torch": latency.get("torch"), "cuda_runtime": latency.get("cuda_runtime"),\n        "checkpoint_sha256": report.get("evaluated_checkpoint_sha256"),\n        "report_path": str(source),\n    }\n\n\ndef emit_report(path):\n    """Print after the evaluator has successfully saved its report."""\n    path = Path(path)\n    report = json.loads(path.read_text(encoding="utf-8"))\n    row = flatten_report(report, path)\n    print(f"\\n=== RESULTS {row[\'dataset\'].upper()} / {row[\'variant\']} / {row[\'mode\']} ===",\n          flush=True)\n    print(f"Primary branch: {row[\'primary_branch\']}; videos: {row[\'video_count\']}", flush=True)\n    if report.get("metrics"):\n        branch_rows = []\n        for name, values in report["metrics"].items():\n            numbers = [values.get("roc_auc"), values.get("ap", values.get("frame_ap")),\n                       values.get("ano_auc")]\n            branch_rows.append([name, "primary" if name == row["primary_branch"] else "auxiliary",\n                                *(format_number(None if n is None else n * 100) for n in numbers)])\n        print(render_table(["Branch", "Role", "ROC-AUC (%)", "AP (%)", "Ano-AUC (%)"],\n                           branch_rows, numeric=(2, 3, 4)), flush=True)\n        print("Historical reproduced V0 reference check (pp):", flush=True)\n        print(render_table(["Metric", "Difference"], [\n            ["ROC-AUC", format_number(row["auc_vs_reference_v0_pp"], 5)],\n            ["AP", format_number(row["ap_vs_reference_v0_pp"], 5)],\n        ], numeric=(1,)), flush=True)\n    else:\n        print("Benchmark: N/A (smoke test; not a full evaluation)", flush=True)\n    print("Latency feature -> logits and memory:", flush=True)\n    print(render_table(["Measurement", "Value", "Unit"], [\n        ["Mean", format_number(row["mean_ms_per_video"], 3), "ms/video"],\n        ["Median", format_number(row["median_ms_per_video"], 3), "ms/video"],\n        ["P95", format_number(row["p95_ms_per_video"], 3), "ms/video"],\n        ["Amortized latency", format_number(row["ms_per_snippet"], 6), "ms/snippet"],\n        ["Throughput", format_number(row["snippets_per_second"], 2), "snippet/s"],\n        ["Peak allocated VRAM", format_number(row["peak_torch_vram_mib"], 2), "MiB"],\n    ], numeric=(1,)), flush=True)\n    parameter_rows = []\n    for label, key in (("Loaded model", "model_parameters"),\n                       ("Additional head", "additional_head_parameters"),\n                       ("Total resident", "total_resident_parameters")):\n        count = row[key]\n        parameter_rows.append([label, "N/A" if count is None else f"{count:,}",\n                               "N/A" if count is None else f"{count / 1e6:.4f}"])\n    print(render_table(["Parameters", "Count", "Millions"], parameter_rows,\n                       numeric=(1, 2)), flush=True)\n    print(render_table(["Run setting", "Value"], [\n        [label, row[key]] for label, key in (\n            ("GPU", "gpu"), ("Input dtype", "input_dtypes"),\n            ("Block length", "visual_block_length"), ("Frames/snippet", "frames_per_feature"),\n            ("Warm-up calls", "warmup"), ("Repeats/video", "repeats"),\n            ("Torch", "torch"), ("CUDA runtime", "cuda_runtime"),\n        )\n    ]), flush=True)\n    print("Offline feature inference; excludes raw video decode and visual CLIP extraction. "\n          "No Base/Novel or online end-to-end metrics in this protocol.", flush=True)\n    print(f"Report: {path}\\n", flush=True)\n    return row\n\n\ndef collect_reports(output):\n    output = Path(output)\n    paths = sorted(output.rglob("run.json"))\n    rows = []\n    reports = []\n    for path in paths:\n        report = json.loads(path.read_text(encoding="utf-8"))\n        if "latency" not in report:\n            continue\n        rows.append(flatten_report(report, path.relative_to(output).as_posix()))\n        reports.append({"path": path.relative_to(output).as_posix(), "report": report})\n    if not rows:\n        return []\n    baseline = {(r["dataset"], r["mode"]): r for r in rows if r["variant"] == "v0"}\n    for row in rows:\n        ref = baseline.get((row["dataset"], row["mode"]))\n        for key, out in (("roc_auc_pct", "auc_vs_same_run_v0_pp"),\n                         ("ap_pct", "ap_vs_same_run_v0_pp"),\n                         ("ano_auc_pct", "ano_auc_vs_same_run_v0_pp")):\n            row[out] = (row[key] - ref[key] if ref and ref.get(key) is not None\n                        and row.get(key) is not None else None)\n        for key, out in (("mean_ms_per_video", "mean_latency_vs_same_run_v0_pct"),\n                         ("ms_per_snippet", "snippet_latency_vs_same_run_v0_pct")):\n            row[out] = (100 * (row[key] / ref[key] - 1) if ref and ref.get(key)\n                        and row.get(key) is not None else None)\n    with (output / "benchmark_summary.csv").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    (output / "benchmark_summary.json").write_text(json.dumps({\n        "protocol": "closed-set WSVAD; no Base/Novel, no FLOPs",\n        "metrics_scale": "percent; differences in percentage points",\n        "rows": rows, "reports": reports,\n    }, indent=2, ensure_ascii=False), encoding="utf-8")\n    print("\\n=== SUMMARY: primary metrics and latency ===", flush=True)\n    print("QUALITY: differences use V0 from the same dataset/mode/run", flush=True)\n    print(render_table(\n        ["Dataset", "Variant", "AUC (%)", "AP (%)", "Ano-AUC (%)", "AUC diff pp", "AP diff pp"],\n        [[r["dataset"], r["variant"], *(format_number(r[key]) for key in (\n            "roc_auc_pct", "ap_pct", "ano_auc_pct", "auc_vs_same_run_v0_pp",\n            "ap_vs_same_run_v0_pp"))] for r in rows], numeric=(2, 3, 4, 5, 6)), flush=True)\n    print("LATENCY: ms/video; difference uses same-run V0", flush=True)\n    print(render_table(\n        ["Dataset", "Variant", "Mean", "Median", "P95", "ms/snippet", "Mean diff (%)"],\n        [[r["dataset"], r["variant"], *(format_number(r[key], 5) for key in (\n            "mean_ms_per_video", "median_ms_per_video", "p95_ms_per_video", "ms_per_snippet",\n            "mean_latency_vs_same_run_v0_pct"))] for r in rows], numeric=(2, 3, 4, 5, 6)), flush=True)\n    print("Saved benchmark_summary.csv and benchmark_summary.json", flush=True)\n    return rows\n\n\ndef save_run_config(path, config, runner):\n    """Persist exact subprocess configuration before it starts, including failed runs."""\n    path = Path(path)\n    receipt = {"config": config, "runner_sha256": hashlib.sha256(runner.encode()).hexdigest()}\n    path.write_text(json.dumps(receipt, default=str, indent=2, ensure_ascii=False), encoding="utf-8")\n'


## Hai evaluator giữ giao thức V0


In [ ]:
UCF_RUNNER = '\nCOMMIT = "c41067f07d252efcda18008bea367886070c33b0"\nUPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"\nEXPECTED_INPUTS = {\n    "checkpoint_sha256": "d064aba05190ce5f6652151e6259b24ca6dda83a1e6289bf343afb6f905a2c79",\n    "split_sha256": "d621f4ccfb3e26274d215ad53b83fd1f596bb7e9f85cde69e01399db3da97c7f",\n    "ground_truth_sha256": "c6448801e0f73d19adf82506cd58d60a90c4f0913e49dcc1ad1d677ee12d6301",\n    "annotation_sha256": "82d581cd4685480ffda23eb0fe1bca441b672bfe99c457adca2df4cf4780c503",\n    "feature_inventory_sha256": "8aa3a3c58f9bbbcc8082e024f3892e014f059374e8b52ca737ae709945210026"\n}\n\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport time\nfrom collections import defaultdict\nfrom shutil import copyfile, copyfileobj, copytree\nfrom urllib.error import URLError\nfrom zipfile import BadZipFile, ZipFile\n\nif not INPUT.is_dir():\n    raise FileNotFoundError("Import notebook trên Kaggle và gắn input trước.")\n\ndef git(*args):\n    return subprocess.check_output(["git", *map(str, args)], text=True).strip()\n\n\nif not SRC_REPO.exists():\n    SRC_REPO.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as checkout_dir:\n        checkout = Path(checkout_dir) / "source"\n        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)\n        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)\n        checkout.rename(SRC_REPO)\nif git("-C", SRC_REPO, "rev-parse", "HEAD") != COMMIT:\n    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} sai commit. Hãy dùng thư mục nguồn mới.")\nif git("-C", SRC_REPO, "status", "--porcelain", "--untracked-files=no"):\n    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} có thay đổi. Hãy dùng checkout sạch.")\n\npackages = {\n    "ftfy": "ftfy", "packaging": "packaging", "regex": "regex",\n    "tqdm": "tqdm", "sklearn": "scikit-learn",\n}\nmissing_packages = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]\nif missing_packages:\n    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)\n\nnp = importlib.import_module("numpy")\n\n# Modern setuptools no longer ships pkg_resources. Adapt the import in a copy.\nstage_directory = tempfile.TemporaryDirectory(prefix="vadclip_ucf_v5_")\nstaging = Path(stage_directory.name)\nruntime_src = staging / "runtime"\ncopytree(SRC_REPO / "src", runtime_src, dirs_exist_ok=True)\nfrom v7_windows import overlap_forward\n\nclip_py = runtime_src / "clip" / "clip.py"\nclip_source = clip_py.read_text(encoding="utf-8")\nold_import = "from pkg_resources import packaging"\nif clip_source.count(old_import) != 1:\n    raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích. Kiểm commit.")\nclip_py.write_text(clip_source.replace(old_import, "import packaging.version"), encoding="utf-8")\n\nOUTPUT.mkdir(parents=True, exist_ok=True)\nprint("Commit VadCLIP", COMMIT)\nprint(f"Chấm UCF-Crime: {VARIANT}")\n\nCLASSES = (\n    "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",\n    "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",\n)\nFRAMES_PER_FEATURE = 16\nFEATURE_DIM = 512\nVIDEO_COUNT = 290\nCLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\n\n\ndef sha256(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef read_split(list_root):\n    """Đọc theo đúng thứ tự repo. Tính khoảng [start, end) theo frame gốc."""\n    with (list_root / "ucf_CLIP_rgbtest.csv").open(newline="", encoding="utf-8") as stream:\n        entries = list(csv.DictReader(stream))\n    annotations = {}\n    for line in (list_root / "Temporal_Anomaly_Annotation.txt").read_text().splitlines():\n        if not line.strip():\n            continue\n        name, label, *bounds = line.split()\n        video_id = Path(name).stem\n        if len(bounds) != 4 or video_id in annotations:\n            raise ValueError(f"Annotation không hợp lệ: {video_id}")\n        intervals = []\n        for start, end in zip(bounds[::2], bounds[1::2]):\n            lo, hi = int(start), int(end)\n            if (lo, hi) == (-1, -1):\n                continue\n            if not 0 <= lo < hi:\n                raise ValueError(f"Khoảng frame không hợp lệ: {video_id} {(lo, hi)}")\n            intervals.append((lo, hi))\n        annotations[video_id] = (label, intervals)\n\n    rows = []\n    for entry in entries:\n        feature_name = Path(entry["path"]).name\n        match = re.fullmatch(r"(.+)__([0-9])\\.npy", feature_name)\n        if match is None:\n            raise ValueError(f"Tên feature sai: {feature_name}")\n        video_id, crop = match.groups()\n        label, intervals = annotations[video_id]\n        if label != entry["label"] or label not in CLASSES:\n            raise ValueError(f"Nhãn không khớp giữa CSV và annotation: {video_id}")\n        rows.append({\n            "id": video_id, "label": label, "feature_name": feature_name,\n            "crop": int(crop), "intervals": intervals,\n        })\n    if len(rows) != VIDEO_COUNT or len({row["id"] for row in rows}) != VIDEO_COUNT:\n        raise ValueError("Cần đúng 290 ID duy nhất từ split UCF-Crime chính thức.")\n    return rows\n\n\n\ndef stage_member(archive, member, destination):\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with archive.open(member) as source, destination.open("wb") as target:\n        copyfileobj(source, target, length=1024 * 1024)\n    return destination\n\n\ndef find_inputs(root, keys):\n    """Tìm theo tên file chính thức, kể cả khi Kaggle giữ ZIP nguyên gói."""\n    found = defaultdict(list)\n    if root.is_file():\n        archive_path = root\n    elif root.is_dir():\n        for folder, _, filenames in os.walk(root):\n            for name in filenames:\n                key = name.casefold()\n                if key in keys:\n                    found[key].append(Path(folder) / name)\n        if found:\n            return found\n        archives = sorted(root.rglob("ucfclipfeatures_test290.zip"))\n        if len(archives) != 1:\n            raise RuntimeError(\n                "Gắn Dataset feature UCF-Crime của V0 hoặc chỉ định DATA_ROOT."\n            )\n        archive_path = archives[0]\n    else:\n        raise FileNotFoundError(f"Không thấy DATA_ROOT: {root}")\n\n    with ZipFile(archive_path) as archive:\n        for index, member in enumerate(archive.infolist()):\n            name = Path(member.filename).name\n            key = name.casefold()\n            if not member.is_dir() and key in keys:\n                destination = staging / "features" / str(index) / name\n                found[key].append(stage_member(archive, member, destination))\n    return found\n\n\ndef checkpoint_candidates():\n    if CHECKPOINT is not None:\n        path = Path(CHECKPOINT)\n        if not path.is_file():\n            raise FileNotFoundError(f"UCF_CHECKPOINT không trỏ đến file đang gắn: {path}")\n        return [path]\n\n    candidates = []\n    for path in INPUT.rglob("*"):\n        if path.suffix.lower() not in {".pth", ".zip"}:\n            continue\n        name = path.stem.casefold()\n        is_ucf_checkpoint = name.startswith("model_ucf") or ("ucf" in name and "checkpoint" in name)\n        if is_ucf_checkpoint and path.is_file():\n            candidates.append(path)\n    return sorted(candidates, key=lambda path: (path.suffix.lower() == ".zip", str(path)))\n\n\ndef choose_checkpoint():\n    expected = EXPECTED_INPUTS["checkpoint_sha256"]\n    candidates = checkpoint_candidates()\n    if not candidates:\n        raise FileNotFoundError(\n            "Không thấy checkpoint UCF trong /kaggle/input. Vào Add Input và gắn Dataset có "\n            "model_ucf.pth hoặc vadclip_ucf_checkpoint_upload.zip. Nếu tên khác, "\n            "đặt UCF_CHECKPOINT thành đường dẫn đầy đủ của file."\n        )\n\n    rejected = []\n    for path in candidates:\n        try:\n            if path.suffix.lower() == ".zip":\n                with ZipFile(path) as archive:\n                    members = [\n                        item for item in archive.infolist()\n                        if not item.is_dir() and Path(item.filename).name.casefold() == "model_ucf.pth"\n                    ]\n                    for member in members:\n                        with archive.open(member) as stream:\n                            digest = hashlib.file_digest(stream, "sha256").hexdigest()\n                        if digest == expected:\n                            print(f"Checkpoint UCF V0 khớp SHA-256: {path} ({member.filename})")\n                            return stage_member(archive, member, staging / "checkpoint" / "model_ucf.pth")\n                rejected.append(f"{path.name} không chứa model_ucf.pth khớp SHA-256")\n            elif sha256(path) == expected:\n                print(f"Checkpoint UCF V0 khớp SHA-256: {path}")\n                return path\n            else:\n                rejected.append(f"{path.name} sai SHA-256")\n        except (BadZipFile, OSError) as exc:\n            rejected.append(f"{path.name}: {exc}")\n    raise ValueError("Không tìm được checkpoint UCF đúng hash V0. " + "; ".join(rejected))\n\n\ndef input_fingerprint(rows, list_root, checkpoint_path):\n    inventory = [[row["id"], row["label"], row["steps"], row["feature_sha256"]] for row in rows]\n    inventory_bytes = json.dumps(inventory, separators=(",", ":")).encode()\n    return {\n        "checkpoint_sha256": sha256(checkpoint_path),\n        "split_sha256": sha256(list_root / "ucf_CLIP_rgbtest.csv"),\n        "ground_truth_sha256": sha256(list_root / "gt_ucf.npy"),\n        "annotation_sha256": sha256(list_root / "Temporal_Anomaly_Annotation.txt"),\n        "feature_inventory_sha256": hashlib.sha256(inventory_bytes).hexdigest(),\n    }\n\n\ndef prepare_clip_cache(source, cache_dir):\n    """Đặt file OpenAI đã xác minh vào đúng cache mà loader gốc đọc."""\n    if not source.is_file():\n        raise FileNotFoundError(f"Không thấy CLIP_WEIGHTS: {source}")\n    if sha256(source) != CLIP_SHA256:\n        raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")\n    target = cache_dir / "ViT-B-16.pt"\n    if target.is_file() and sha256(target) == CLIP_SHA256:\n        return target\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    temporary = cache_dir / "ViT-B-16.pt.vadclip-part"\n    copyfile(source, temporary)\n    if sha256(temporary) != CLIP_SHA256:\n        raise ValueError("Bản sao CLIP lỗi. Kiểm dung lượng và file input.")\n    temporary.replace(target)\n    return target\n\n\ndef feature_array(path):\n    array = np.load(path, mmap_mode="r", allow_pickle=False)\n    if array.ndim != 2 or array.shape[1] != FEATURE_DIM or array.shape[0] < 1:\n        raise ValueError(f"Feature cần dạng [T, 512], nhận {array.shape}: {path}")\n    if array.dtype not in (np.float16, np.float32, np.float64):\n        raise ValueError(f"Feature phải là số thực, nhận {array.dtype}: {path}")\n    return array\n\n\ndef frame_labels(row, length):\n    labels = np.zeros(length, dtype=np.uint8)\n    for start, end in row["intervals"]:\n        labels[start:end] = 1\n    return labels\n\n\ndef validate_ground_truth(rows, gt):\n    """Kiểm cả nội dung và thứ tự, ngoài phép kiểm tổng số frame."""\n    offset = 0\n    for row in rows:\n        stop = offset + row["steps"] * FRAMES_PER_FEATURE\n        expected = frame_labels(row, stop - offset)\n        if not np.array_equal(gt[offset:stop], expected):\n            raise ValueError(\n                f"Ground truth lệch tại {row[\'id\']} (frame offset {offset}, "\n                f"{row[\'steps\']} bước). Kiểm lại bản video/feature và frame count."\n            )\n        offset = stop\n    if offset != len(gt):\n        raise ValueError(f"Số frame không khớp: input={offset}, ground truth={len(gt)}")\n\n# Kiểm đầu vào và đối chiếu hash với V0 đã chạy.\nlist_root = SRC_REPO / "list"\nrows = read_split(list_root)\ndata_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT\nexpected = {row["feature_name"].casefold(): row["feature_name"] for row in rows}\nfound = find_inputs(data_root, expected)\nmissing = sorted(expected[key] for key in expected if not found[key])\nambiguous = {\n    expected[key]: [str(path) for path in sorted(found[key])]\n    for key in expected if len(found[key]) > 1\n}\nif missing or ambiguous:\n    issue_path = OUTPUT / "input_issues.json"\n    issue_path.write_text(json.dumps({\n        "missing": missing,\n        "ambiguous": ambiguous,\n    }, indent=2), encoding="utf-8")\n    raise RuntimeError(\n        f"Thiếu {len(missing)} file, trùng {len(ambiguous)} ID. "\n        f"Xem {issue_path} và kiểm DATA_ROOT."\n    )\nfor row in rows:\n    path = found[row["feature_name"].casefold()][0]\n    row["path"] = path\n    row["steps"] = len(feature_array(path))\n    row["feature_sha256"] = sha256(path)\n\ngt = np.load(list_root / "gt_ucf.npy", allow_pickle=False)\nif gt.ndim != 1 or set(np.unique(gt)) != {0.0, 1.0}:\n    raise ValueError("Ground truth UCF cần vector nhị phân cấp frame.")\nvalidate_ground_truth(rows, gt)\ncheckpoint_path = choose_checkpoint()\ninput_hashes = input_fingerprint(rows, list_root, checkpoint_path)\nmismatches = [key for key, value in EXPECTED_INPUTS.items() if input_hashes[key] != value]\nif mismatches:\n    raise ValueError("Đầu vào khác lượt V0 đã đối chiếu: " + ", ".join(mismatches))\nprint(f"Đã khớp {len(rows)} video, {len(gt):,} frame và toàn bộ hash đầu vào với V0.")\n\nimport torch\n\nsys.path.insert(0, str(runtime_src))\nfrom model import CLIPVAD\nfrom ucf_option import parser as ucf_parser\nfrom utils.tools import get_batch_mask\n\nif not torch.cuda.is_available():\n    raise RuntimeError("Hãy bật GPU trong Kaggle. Mã VadCLIP gốc cần CUDA khi chạy forward.")\ndevice = torch.device("cuda:0")\ntorch.cuda.set_device(device)\ntorch.manual_seed(234)\n\n# Dùng đúng cấu hình UCF từ mã tác giả, không đọc cờ của kernel Jupyter.\nargs = ucf_parser.parse_args([])\nevaluated_checkpoint = checkpoint_path\nevaluated_sha256 = sha256(evaluated_checkpoint)\nstate = torch.load(evaluated_checkpoint, map_location="cpu", weights_only=True)\nif isinstance(state, dict):\n    for field in ("state_dict", "model_state_dict"):\n        if field in state:\n            state = state[field]\n            break\nif CLIP_WEIGHTS is not None:\n    prepare_clip_cache(Path(CLIP_WEIGHTS), Path.home() / ".cache" / "clip")\ntry:\n    model = CLIPVAD(\n        args.classes_num, args.embed_dim, args.visual_length, args.visual_width, args.visual_head,\n        args.visual_layers, args.attn_window, args.prompt_prefix, args.prompt_postfix, device,\n    )\nexcept URLError as exc:\n    raise RuntimeError(\n        "Không tải được CLIP ViT-B/16. Bật Internet hoặc gắn ViT-B-16.pt chính thức "\n        "và đặt CLIP_WEIGHTS bằng đường dẫn file."\n    ) from exc\ntry:\n    model.load_state_dict(state, strict=True)\nexcept (RuntimeError, TypeError) as exc:\n    raise RuntimeError(\n        f"Checkpoint không khớp model VadCLIP UCF-Crime: {evaluated_checkpoint}. "\n        "Kiểm lại file từ cột Model dòng UCF-Crime, khác file ViT-B-16.pt."\n    ) from exc\ndel state\nmodel.to(device).eval()\nprint("Đã nạp đầy đủ checkpoint trên", torch.cuda.get_device_name(device))\n\nfrom latency_inference import InferenceLatency\n\nlatency = InferenceLatency(LATENCY_WARMUP, LATENCY_REPEATS)\n\n\ndef predict_video(feature, model, device, block_length):\n    """Trả điểm [T, 2], lần lượt sigmoid(C) và 1-softmax(A)[Normal]."""\n    steps = len(feature)\n    if steps < 1 or not np.isfinite(feature).all():\n        raise ValueError("Feature rỗng hoặc chứa NaN/Inf.")\n    blocks = (steps + block_length - 1) // block_length\n    padded = np.zeros((blocks * block_length, FEATURE_DIM), dtype=np.float32)\n    padded[:steps] = feature\n    lengths = torch.tensor([\n        min(block_length, steps - i * block_length) for i in range(blocks)\n    ])\n    visual = torch.from_numpy(padded.reshape(blocks, block_length, FEATURE_DIM)).to(device)\n    mask = get_batch_mask(lengths, block_length).to(device)\n    with torch.no_grad():\n        def dispatch():\n            if VARIANT == "v7":\n                t, logits_c, logits_a = overlap_forward(\n                    model, visual, mask, list(CLASSES), lengths, get_batch_mask,\n                )\n            else:\n                t, logits_c, logits_a = model(visual, mask, list(CLASSES), lengths)\n            return t, logits_c, logits_a\n\n        _, logits_c, logits_a = latency.measure(\n            dispatch, device, len(latency.rows) // LATENCY_REPEATS,\n            steps, visual.shape, visual.dtype,\n        )\n        c_score = logits_c.reshape(-1)[:steps].sigmoid()\n        a_score = 1 - logits_a.reshape(-1, len(CLASSES))[:steps].softmax(dim=-1)[:, 0]\n        scores = torch.stack((c_score, a_score), dim=1).cpu().numpy()\n    if scores.shape != (steps, 2) or not np.isfinite(scores).all():\n        raise ValueError(f"Điểm bất thường không hợp lệ: {scores.shape}")\n    return scores\n\n\ndef run_video(row):\n    feature = feature_array(row["path"])\n    if feature.shape != (row["steps"], FEATURE_DIM):\n        raise ValueError(f"Feature {row[\'id\']} sai shape: {feature.shape}")\n    scores = predict_video(feature, model, device, args.visual_length)\n    for timing in latency.rows[-LATENCY_REPEATS:]:\n        timing["video_id"] = row["id"]\n    return scores\n\n\n# Chọn theo thứ tự split và độ dài, trước khi xem điểm.\npilot = sorted({\n    next(i for i, row in enumerate(rows) if row["label"] != "Normal"),\n    next(i for i, row in enumerate(rows) if row["label"] == "Normal"),\n    max(range(len(rows)), key=lambda i: rows[i]["steps"]),\n})\ntorch.cuda.reset_peak_memory_stats(device)\nstarted = time.perf_counter()\nscores_by_index = {}\nfor i in pilot:\n    before = time.perf_counter()\n    scores_by_index[i] = run_video(rows[i])\n    print(rows[i]["id"], scores_by_index[i].shape, f"{time.perf_counter() - before:.2f}s")\nsmoke_seconds = time.perf_counter() - started\nprint(f"Smoke test hoàn tất trong {smoke_seconds:.2f}s.")\n\nif not SMOKE_ONLY:\n    for i, row in enumerate(rows):\n        if i not in scores_by_index:\n            scores_by_index[i] = run_video(row)\n        if (i + 1) % 25 == 0 or i + 1 == len(rows):\n            print(f"Đã chạy {i + 1}/{len(rows)} video", flush=True)\n\nfrom importlib.metadata import version\n\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nSCORE_COLUMNS = ("classification", "alignment")\n\n\n\n\ndef score_benchmark(rows, snippet_scores, gt):\n    steps_per_video = np.asarray([row["steps"] for row in rows])\n    if snippet_scores.shape != (sum(steps_per_video), len(SCORE_COLUMNS)):\n        raise ValueError("Số điểm snippet hoặc cột không khớp.")\n    frame_scores = np.repeat(snippet_scores, FRAMES_PER_FEATURE, axis=0)\n    if len(frame_scores) != len(gt):\n        raise ValueError("Số điểm cấp frame không khớp ground truth.")\n    frame_mask = np.repeat(\n        [row["label"] != "Normal" for row in rows],\n        steps_per_video * FRAMES_PER_FEATURE,\n    )\n    metrics = {}\n    for column, name in enumerate(SCORE_COLUMNS):\n        scores = frame_scores[:, column]\n        metrics[name] = {\n            "roc_auc": float(roc_auc_score(gt, scores)),\n            "ap": float(average_precision_score(gt, scores)),\n            "ano_auc": float(roc_auc_score(gt[frame_mask], scores[frame_mask])),\n        }\n    return metrics\n\n\nselected = sorted(scores_by_index)\nif not SMOKE_ONLY and selected != list(range(len(rows))):\n    raise ValueError("Thiếu dự đoán. Cần chạy toàn bộ split trước khi chấm.")\nselected_rows = [rows[i] for i in selected]\nordered = [scores_by_index[i] for i in selected]\nif any(score.shape != (row["steps"], 2)\n       for row, score in zip(selected_rows, ordered, strict=True)):\n    raise ValueError("Số điểm từng video không khớp.")\nsteps_per_video = [row["steps"] for row in selected_rows]\nscores = np.concatenate(ordered, axis=0)\n\noutput_dir = OUTPUT / ("smoke" if SMOKE_ONLY else "full")\noutput_dir.mkdir(parents=True, exist_ok=True)\nlatency_summary = latency.save(\n    output_dir, "ucf", VARIANT, "smoke" if SMOKE_ONLY else "full", device,\n)\nfrom benchmark_reporting import count_parameters\n\nmetadata = {\n    "parameters": count_parameters(model, None),\n    "run_kind": "smoke_inference" if SMOKE_ONLY else "full_checkpoint_inference",\n    "benchmark_scope": "ucf_crime_official_test",\n    "evaluation_status": "exploratory_on_previously_seen_official_test",\n    "commit": COMMIT,\n    "runtime_patch": "CLIP imports packaging.version directly",\n    **input_hashes,\n    "clip_backbone_sha256": CLIP_SHA256,\n    "frames_per_feature": FRAMES_PER_FEATURE,\n    "visual_block_length": args.visual_length,\n    "label_order": CLASSES,\n    "score_columns": SCORE_COLUMNS,\n    "evaluated_checkpoint_sha256": evaluated_sha256,\n    "python": sys.version.split()[0],\n    "libraries": {name: version(name) for name in (\n        "torch", "torchvision", "numpy", "scipy", "scikit-learn", "pillow",\n    )},\n    "cuda": torch.version.cuda,\n    "gpu": torch.cuda.get_device_name(device),\n    "latency": latency_summary,\n    "smoke_seconds": smoke_seconds,\n    "inference_seconds": time.perf_counter() - started,\n    "peak_torch_vram_mib": torch.cuda.max_memory_allocated(device) / (1024 ** 2),\n    "video_count": len(selected_rows),\n    "videos": [{key: value for key, value in row.items() if key != "path"}\n               for row in selected_rows],\n    "metrics": None,\n}\nif not SMOKE_ONLY:\n    metrics = score_benchmark(rows, scores, gt)\n    metadata["metrics"] = metrics\n    print("UCF metrics saved; formatted table follows.", flush=True)\n\nnp.savez_compressed(\n    output_dir / "predictions.npz",\n    video_id=np.asarray([row["id"] for row in selected_rows]),\n    video_label=np.asarray([row["label"] for row in selected_rows]),\n    snippet_offsets=np.cumsum([0] + steps_per_video),\n    score_columns=np.asarray(SCORE_COLUMNS),\n    scores=scores,\n    frame_gt=gt if not SMOKE_ONLY else np.empty(0, dtype=np.uint8),\n)\n(output_dir / "run.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\nstage_directory.cleanup()\nprint("Đầu ra", output_dir)\n'


In [ ]:
XD_RUNNER = '\nCOMMIT = "c41067f07d252efcda18008bea367886070c33b0"\nUPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"\nVIDEO_COUNT = 800\nFRAMES_PER_FEATURE = 16\nFRAME_COUNT = 2_331_296\nFEATURE_DIM = 512\nPAPER_AP = 0.8451\nPACK_MANIFEST_NAME = "xd_test_manifest.json"\nPACK_ARCHIVE_NAME = "vadclip_xd_test800.zip"\nPREVIOUS_PACK_ARCHIVE_NAME = "vadclip_xd_test800_kaggle_v2.zip"\nPACK_MANIFEST_SHA256 = "8e5d329708857338fa94fb63a1d5733d9d8536250e6c09c3ba012f54dce997b2"\nCLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"\n\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom collections import defaultdict\nfrom contextlib import ExitStack\nfrom pathlib import PurePosixPath\nfrom zipfile import BadZipFile, ZipFile\n\nif RUN_MODE not in {"smoke", "full"}:\n    raise ValueError("Evaluator XD cần chế độ smoke hoặc full.")\nif not INPUT.is_dir():\n    raise FileNotFoundError("Gắn dữ liệu qua Add Input trong Kaggle trước khi chạy.")\n\nOUTPUT.mkdir(parents=True, exist_ok=True)\nscratch = SRC_REPO.parent\nscratch.mkdir(parents=True, exist_ok=True)\nsrc_repo = SRC_REPO\nif not src_repo.exists():\n    with tempfile.TemporaryDirectory(dir=scratch) as checkout_dir:\n        checkout = Path(checkout_dir) / "source"\n        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)\n        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)\n        checkout.rename(src_repo)\n\ndef git(*args):\n    return subprocess.check_output(["git", "-C", str(src_repo), *args], text=True).strip()\n\nif git("rev-parse", "HEAD") != COMMIT or git("status", "--porcelain", "--untracked-files=all"):\n    raise RuntimeError(f"Cần checkout sạch tại commit {COMMIT}. Kiểm tra {src_repo}.")\n\nnp = importlib.import_module("numpy")\n\ndef sha256(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef zip_members(path):\n    try:\n        with ZipFile(path) as archive:\n            return [name for name in archive.namelist() if not name.endswith("/")]\n    except BadZipFile as exc:\n        raise ValueError(f"ZIP chưa tải đủ hoặc bị hỏng: {path.name}") from exc\n\n\ndef basename(value):\n    return PurePosixPath(str(value).replace(chr(92), "/")).name\n\n\ndef read_split(path):\n    with path.open(newline="", encoding="utf-8-sig") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames is None or not {"path", "label"} <= set(reader.fieldnames):\n            raise ValueError("CSV tác giả cần cột path và label.")\n        fields, rows = reader.fieldnames, list(reader)\n    source_paths = [row["path"] for row in rows]\n    if (\n        len(rows) != VIDEO_COUNT\n        or len(set(source_paths)) != VIDEO_COUNT\n        or any(not path or not basename(path).endswith("__0.npy") for path in source_paths)\n        or any(not row["label"] for row in rows)\n    ):\n        raise ValueError(f"Split XD phải gồm {VIDEO_COUNT} dòng feature hợp lệ, duy nhất.")\n    return fields, rows\n\n\ndef matching_files(root, expected):\n    found = defaultdict(list)\n    if FEATURE_ZIP is None:\n        if not root.is_dir():\n            raise FileNotFoundError(f"DATA_ROOT không tồn tại: {root}")\n        for path in root.rglob("*.npy"):\n            if path.name in expected:\n                found[path.name].append((path, None))\n        if found:\n            return found\n        archives = sorted(root.rglob("*.zip"))\n    else:\n        archives = [Path(FEATURE_ZIP)]\n\n    for archive_path in archives:\n        if not archive_path.is_file():\n            raise FileNotFoundError(f"Không thấy FEATURE_ZIP: {archive_path}")\n        for member in zip_members(archive_path):\n            name = basename(member)\n            if name in expected:\n                found[name].append((archive_path, member))\n    return found\n\n\ndef match_one(original, candidates):\n    # Nếu hai file cùng tên, chỉ phân giải khi hậu tố thư mục chính thức khớp duy nhất.\n    if len(candidates) <= 1:\n        return candidates\n    official = PurePosixPath(original.replace(chr(92), "/")).parts\n    for depth in range(2, len(official) + 1):\n        matches = [\n            (path, member)\n            for path, member in candidates\n            if PurePosixPath((member or path.as_posix()).replace(chr(92), "/")).parts[-depth:]\n            == official[-depth:]\n        ]\n        if len(matches) == 1:\n            return matches\n        if not matches:\n            break\n    return candidates\n\n\ndef pick_checkpoint(root):\n    if CHECKPOINT is not None:\n        path = Path(CHECKPOINT)\n        if not path.is_file() or path.suffix.lower() not in {".pth", ".zip"}:\n            raise FileNotFoundError(f"CHECKPOINT cần đường dẫn .pth hoặc .zip hợp lệ: {path}")\n        return path\n    weights = [path for path in root.rglob("*.pth") if path.name in {"model_xd.pth", "model.pth"}]\n    if len(weights) == 1:\n        return weights[0]\n    if not weights:\n        archives = [\n            path for path in root.rglob("*.zip")\n            if any(\n                basename(member) in {"model_xd.pth", "model.pth"}\n                for member in zip_members(path)\n            )\n        ]\n        if len(archives) == 1:\n            return archives[0]\n    raise RuntimeError(\n        "Gắn checkpoint VadCLIP XD-Violence. Nếu tên khác hoặc có nhiều bản, "\n        "đặt CHECKPOINT bằng đường dẫn file .pth hoặc ZIP chỉ chứa một .pth."\n    )\n\n\ndef stage_member(archive, member, target):\n    # Chỉ ghi tới tên đích do notebook tạo. Không giải nén theo đường dẫn trong ZIP.\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with archive.open(member) as src, target.open("wb") as dest:\n        shutil.copyfileobj(src, dest, length=1024 * 1024)\n    return target\n\n\ndef checkpoint_from_input(root, staging):\n    path = pick_checkpoint(root)\n    if path.suffix.lower() == ".zip":\n        members = [name for name in zip_members(path) if name.lower().endswith(".pth")]\n        if len(members) != 1:\n            raise ValueError(f"ZIP checkpoint phải chứa đúng một .pth: {path.name}")\n        if basename(members[0]) == "model_ucf.pth":\n            raise ValueError("ZIP đang chứa checkpoint UCF-Crime.")\n        with ZipFile(path) as archive:\n            path = stage_member(archive, members[0], staging / "checkpoint.pth")\n    if path.name == "model_ucf.pth" or path.stat().st_size == 0:\n        raise ValueError(f"File checkpoint không hợp lệ cho XD: {path.name}")\n    with path.open("rb") as stream:\n        if stream.read(4) == b"PK\\x03\\x04":\n            zip_members(path)  # xác nhận có thư mục ZIP ở cuối file PyTorch\n    return path\n\n\ndef feature_info(path):\n    array = np.load(path, mmap_mode="r", allow_pickle=False)\n    if (\n        array.ndim != 2 or array.shape[0] == 0 or array.shape[1] != FEATURE_DIM\n        or array.dtype not in (np.float16, np.float32, np.float64)\n        or not np.isfinite(array).all()\n    ):\n        raise ValueError(f"Feature cần mảng số thực hữu hạn [T, {FEATURE_DIM}]: {path.name}")\n    return len(array), sha256(path)\n\n\ndef read_pack_manifest(root):\n    candidates = [path.read_bytes() for path in root.rglob(PACK_MANIFEST_NAME)]\n    archives = set(root.rglob(PACK_ARCHIVE_NAME)) | set(root.rglob(PREVIOUS_PACK_ARCHIVE_NAME))\n    if FEATURE_ZIP is not None:\n        archives.add(Path(FEATURE_ZIP))\n    for archive_path in sorted(archives):\n        with ZipFile(archive_path) as archive:\n            members = [\n                name for name in archive.namelist() if basename(name) == PACK_MANIFEST_NAME\n            ]\n            candidates.extend(archive.read(name) for name in members)\n    if not candidates:\n        raise FileNotFoundError(\n            "Chưa thấy manifest. Gắn Dataset VadCLIP XD-Violence Test 800 "\n            f"từ {PACK_ARCHIVE_NAME} và giữ nguyên tên file ZIP."\n        )\n    if any(hashlib.sha256(raw).hexdigest() != PACK_MANIFEST_SHA256 for raw in candidates):\n        raise ValueError("Manifest trên Kaggle không khớp bản đã khóa trong notebook.")\n\n    manifest = json.loads(candidates[0])\n    if manifest["upstream_commit"] != COMMIT:\n        raise ValueError("Commit trong manifest không đúng.")\n    return manifest\n\n\ndef check_pack_manifest(manifest, inventory, checkpoint_hash):\n    if manifest["checkpoint_sha256"] != checkpoint_hash:\n        raise ValueError("Sai checkpoint so với manifest đã khóa.")\n    actual_features = {\n        basename(row["official_path"]): {"steps": row["steps"], "sha256": row["feature_sha256"]}\n        for row in inventory\n    }\n    if actual_features != manifest["features"]:\n        names = actual_features.keys() | manifest["features"].keys()\n        first_difference = next(\n            name for name in sorted(names)\n            if actual_features.get(name) != manifest["features"].get(name)\n        )\n        raise ValueError(f"Feature trên Kaggle lệch bản đã khóa: {first_difference}")\n    return manifest["source_features_sha256"]\n\n\ndef prepare_inputs(split, gt_path, data_root, staging, upload_names=None):\n    fields, rows = read_split(split)\n    original_names = {basename(row["path"]) for row in rows}\n    if upload_names is not None and (\n        len(original_names) != VIDEO_COUNT\n        or upload_names.keys() != original_names\n        or len(set(upload_names.values())) != VIDEO_COUNT\n    ):\n        raise ValueError("Ánh xạ tên file không khớp danh sách video trong CSV gốc.")\n    names = upload_names if upload_names is not None else {name: name for name in original_names}\n    found = matching_files(data_root, set(names.values()))\n    resolved = []\n    issues = {"missing": [], "ambiguous": []}\n    for row in rows:\n        original = row["path"]\n        candidates = match_one(original, found[names[basename(original)]])\n        if len(candidates) == 0:\n            issues["missing"].append(original)\n        elif len(candidates) > 1:\n            issues["ambiguous"].append({\n                "expected": original,\n                "candidates": [str(path) if member is None else f"{path.name}::{member}"\n                               for path, member in candidates],\n            })\n        else:\n            resolved.append(candidates[0])\n    if any(issues.values()) or len(set(resolved)) != VIDEO_COUNT:\n        report = OUTPUT / "input_issues.json"\n        report.write_text(json.dumps(issues, ensure_ascii=False, indent=2), encoding="utf-8")\n        raise RuntimeError(\n            f"Thiếu {len(issues[\'missing\'])} file, trùng {len(issues[\'ambiguous\'])} dòng "\n            f"hoặc dùng lại cùng feature. Xem {report}. Kiểm DATA_ROOT/FEATURE_ZIP."\n        )\n\n    gt = np.load(gt_path, allow_pickle=False)\n    if gt.shape != (FRAME_COUNT,) or not np.array_equal(np.unique(gt), [0, 1]):\n        raise ValueError("Ground truth XD sai shape hoặc không phải vector nhị phân.")\n\n    inventory = []\n    with ExitStack() as stack:\n        archives = {}\n        for index, (row, (path, member)) in enumerate(zip(rows, resolved, strict=True)):\n            if member is not None:\n                if path not in archives:\n                    archives[path] = stack.enter_context(ZipFile(path))\n                target = staging / f"{index:04d}" / basename(member)\n                path = stage_member(archives[path], member, target)\n            count, digest = feature_info(path)\n            inventory.append({\n                "official_path": row["path"], "label": row["label"],\n                "steps": count, "feature_sha256": digest,\n            })\n            row["path"] = str(path)\n    steps = sum(item["steps"] for item in inventory)\n    if steps * FRAMES_PER_FEATURE != len(gt):\n        raise ValueError(\n            f"Feature có {steps:,} bước, cần {len(gt) // FRAMES_PER_FEATURE:,} "\n            f"theo ground truth tác giả. Kiểm phiên bản gói feature và crop."\n        )\n\n    offset = 0\n    for item in inventory:\n        stop = offset + item["steps"] * FRAMES_PER_FEATURE\n        positive_frames = int(np.count_nonzero(gt[offset:stop]))\n        if "_label_A" in basename(item["official_path"]) and positive_frames:\n            raise ValueError(f"Lệch ground truth trong video Normal: {item[\'official_path\']}")\n        item["positive_frames"] = positive_frames\n        offset = stop\n\n    resolved_csv = staging / "xd_CLIP_rgbtest_kaggle.csv"\n    with resolved_csv.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(rows)\n    return resolved_csv, inventory, gt\n\n# Preflight CPU. Mỗi lần chạy tạo thư mục mới, không ghi đè input đã gắn.\nstage_directory = tempfile.TemporaryDirectory(prefix="run_", dir=scratch)\nstaging = Path(stage_directory.name)\nsplit = src_repo / "list" / "xd_CLIP_rgbtest.csv"\ngt_path = src_repo / "list" / "gt.npy"\ndata_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT\ntry:\n    pack_manifest = read_pack_manifest(INPUT) if REQUIRE_PACK_MANIFEST else None\n    upload_names = pack_manifest["upload_names"] if pack_manifest is not None else None\n    resolved_csv, inventory, gt = prepare_inputs(split, gt_path, data_root, staging, upload_names)\n    checkpoint_path = checkpoint_from_input(INPUT, staging)\n    checkpoint_hash = sha256(checkpoint_path)\n    source_features_hash = None\n    if pack_manifest is not None:\n        source_features_hash = check_pack_manifest(pack_manifest, inventory, checkpoint_hash)\nexcept Exception:\n    stage_directory.cleanup()\n    raise\n\nreceipt = {\n    "upstream_commit": COMMIT,\n    "split_sha256": sha256(split),\n    "ground_truth_sha256": sha256(gt_path),\n    "checkpoint_sha256": checkpoint_hash,\n    "pack_manifest_sha256": PACK_MANIFEST_SHA256 if REQUIRE_PACK_MANIFEST else None,\n    "source_features_archive_sha256": source_features_hash,\n    "video_count": len(inventory),\n    "snippet_count": sum(row["steps"] for row in inventory),\n    "frame_count": len(gt),\n    "frames_per_feature": FRAMES_PER_FEATURE,\n    "videos": inventory,\n}\nif REFERENCE_MANIFEST is not None:\n    try:\n        reference = json.loads(Path(REFERENCE_MANIFEST).read_text(encoding="utf-8"))\n        for key in ("upstream_commit", "split_sha256", "ground_truth_sha256",\n                    "checkpoint_sha256", "videos"):\n            if receipt[key] != reference.get(key):\n                raise ValueError(f"Bản Kaggle lệch bản tham chiếu ở trường {key}.")\n    except Exception:\n        stage_directory.cleanup()\n        raise\nreceipt["matched_reference_manifest"] = REFERENCE_MANIFEST is not None\n\n(OUTPUT / "preflight.json").write_text(\n    json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8"\n)\nprint(\n    f"Preflight đạt: {len(inventory)} video, {receipt[\'snippet_count\']:,} bước, "\n    f"{len(gt):,} frame."\n)\nprint(f"Checkpoint {checkpoint_path.name}, SHA-256 {checkpoint_hash}")\nif REQUIRE_PACK_MANIFEST:\n    print("SHA-256: khớp cả 800 feature và checkpoint công bố với hai gói đã chuẩn bị.")\n\nreceipt["evaluated_checkpoint_sha256"] = checkpoint_hash\n\nimport contextlib\nimport runpy\nimport time\nimport traceback\nfrom unittest.mock import patch\n\n\n\n\ndef run_upstream():\n    full_run = RUN_MODE == "full"\n    packages = {\n        "ftfy": "ftfy", "packaging": "packaging", "regex": "regex", "tqdm": "tqdm",\n        "sklearn": "scikit-learn", "scipy": "scipy", "pandas": "pandas",\n    }\n    missing = [\n        package for module, package in packages.items()\n        if importlib.util.find_spec(module) is None\n    ]\n    if missing:\n        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)\n\n    import torch\n    from sklearn import metrics\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Bật Accelerator GPU T4 trong Kaggle để chạy inference.")\n    torch.cuda.set_device(0)\n    torch.manual_seed(234)\n    runtime = staging / "runtime"\n    shutil.copytree(src_repo / "src", runtime)\n    from v7_windows import overlap_forward\n\n    clip_source = runtime / "clip" / "clip.py"\n    text = clip_source.read_text(encoding="utf-8")\n    old_import = "from pkg_resources import packaging"\n    if text.count(old_import) != 1:\n        raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích tại commit đã khóa.")\n    clip_source.write_text(text.replace(old_import, "import packaging.version"), encoding="utf-8")\n\n    if CLIP_WEIGHTS is not None:\n        source = Path(CLIP_WEIGHTS)\n        if not source.is_file() or sha256(source) != CLIP_SHA256:\n            raise ValueError("CLIP_WEIGHTS cần file OpenAI ViT-B-16.pt đúng SHA-256.")\n        cache = Path.home() / ".cache" / "clip"\n        cache.mkdir(parents=True, exist_ok=True)\n        target = cache / "ViT-B-16.pt"\n        if source.resolve() != target.resolve():\n            with tempfile.TemporaryDirectory(dir=cache) as copy_dir:\n                temporary = Path(copy_dir) / target.name\n                shutil.copyfile(source, temporary)\n                if sha256(temporary) != CLIP_SHA256:\n                    raise ValueError("Bản sao trọng số CLIP không khớp SHA-256.")\n                temporary.replace(target)\n\n    output_root = OUTPUT / RUN_MODE\n    output_root.mkdir(exist_ok=True)\n    output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))\n    print(f"Đang chạy evaluator gốc. Log: {output_dir / \'upstream.log\'}", flush=True)\n    native_ap = metrics.average_precision_score\n    ap_calls = []\n    upstream_completed = False\n    post_ap_error = None\n    checkpoint_loaded = False\n    forwards = 0\n    torch_load = torch.load\n\n    class SmokeComplete(Exception):\n        pass\n\n    def load_checkpoint(file, *args, **kwargs):\n        nonlocal checkpoint_loaded\n        if not isinstance(file, (str, os.PathLike)) or Path(file).resolve() != checkpoint_path.resolve():\n            return torch_load(file, *args, **kwargs)\n        checkpoint_loaded = True\n        state = torch_load(file, map_location="cpu", weights_only=True)\n        for field in ("state_dict", "model_state_dict"):\n            if isinstance(state, dict) and field in state:\n                return state[field]\n        return state\n\n    def record_ap(y_true, y_score, *args, **kwargs):\n        value = native_ap(y_true, y_score, *args, **kwargs)\n        if np.asarray(y_true).shape == gt.shape and np.array_equal(y_true, gt):\n            scores = np.asarray(y_score).reshape(-1)\n            if scores.shape != gt.shape or not np.isfinite(scores).all():\n                raise ValueError("Điểm frame từ evaluator XD sai shape hoặc chứa NaN/Inf.")\n            grouped = scores.reshape(-1, FRAMES_PER_FEATURE)\n            if not np.all(grouped == grouped[:, :1]):\n                raise ValueError("Điểm từ evaluator không theo quy tắc lặp 16 frame.")\n            ap_calls.append((float(value), grouped[:, 0].copy()))\n        return value\n\n    original_cwd = Path.cwd()\n    original_path = sys.path.copy()\n    original_argv = sys.argv.copy()\n    original_bytecode = sys.dont_write_bytecode\n    try:\n        os.chdir(runtime)\n        sys.path.insert(0, str(runtime))\n        sys.dont_write_bytecode = True\n        from model import CLIPVAD\n        from utils.tools import get_batch_mask\n        from xd_option import parser\n\n        # Giữ các giá trị mặc định XD của tác giả, chỉ thay đường dẫn input.\n        destinations = {action.dest for action in parser._actions}\n        overrides = {\n            "test_list": str(resolved_csv),\n            "gt_path": str(gt_path),\n            "model_path": str(checkpoint_path),\n        }\n        defaults = parser.parse_args([])\n        for key in ("gt_segment_path", "gt_label_path"):\n            default = Path(getattr(defaults, key))\n            path = (src_repo / "src" / default).resolve()\n            if not path.is_file():\n                path = (src_repo / default).resolve()\n            relative = path.relative_to(src_repo)\n            git("ls-files", "--error-unmatch", relative.as_posix())\n            overrides[key] = str(path)\n        if not overrides.keys() <= destinations:\n            raise RuntimeError(f"CLI upstream thiếu tham số: {overrides.keys() - destinations}")\n        flags = []\n        for action in parser._actions:\n            if action.dest in overrides:\n                flags.extend((action.option_strings[0], overrides[action.dest]))\n\n        from latency_inference import InferenceLatency\n\n        latency = InferenceLatency(LATENCY_WARMUP, LATENCY_REPEATS)\n        from benchmark_reporting import count_parameters\n\n        parameter_counts = {}\n        original_forward = CLIPVAD.forward\n\n        smoke_forwards = max(3, next(\n            i for i, row in enumerate(inventory, 1) if row["steps"] > defaults.visual_length\n        ))\n\n        def count_forward(self, images, padding_mask, text, lengths):\n            nonlocal forwards\n            if forwards == 0:\n                parameter_counts.update(count_parameters(self, None))\n\n            def forward(x, mask, prompts, sizes):\n                return original_forward(self, x, mask, prompts, sizes)\n\n            def dispatch():\n                if VARIANT == "v7":\n                    result = overlap_forward(forward, images, padding_mask, text, lengths, get_batch_mask)\n                else:\n                    result = forward(images, padding_mask, text, lengths)\n                return result\n\n            row = inventory[forwards]\n            result = latency.measure(\n                dispatch, images.device, row["official_path"], row["steps"],\n                images.shape, images.dtype,\n            )\n            forwards += 1\n            if RUN_MODE == "smoke" and forwards == smoke_forwards:\n                if any(not torch.isfinite(tensor).all() for tensor in result):\n                    raise ValueError("Smoke test sinh NaN/Inf.")\n                raise SmokeComplete\n            return result\n\n        sys.argv = [str(runtime / "xd_test.py"), *flags]\n        torch.cuda.reset_peak_memory_stats()\n        started = time.perf_counter()\n        with (\n            (output_dir / "upstream.log").open("w", encoding="utf-8") as log,\n            contextlib.redirect_stdout(log),\n            contextlib.redirect_stderr(log),\n            patch.object(torch, "load", load_checkpoint),\n            patch.object(metrics, "average_precision_score", record_ap),\n            patch.object(CLIPVAD, "forward", count_forward),\n        ):\n            try:\n                runpy.run_path(str(runtime / "xd_test.py"), run_name="__main__")\n                upstream_completed = True\n            except SmokeComplete:\n                if RUN_MODE != "smoke":\n                    raise\n            except Exception as exc:\n                if not full_run or len(ap_calls) != 2:\n                    raise\n                traceback.print_exc(file=log)\n                post_ap_error = f"{type(exc).__name__}: {exc}"\n        elapsed = time.perf_counter() - started\n    finally:\n        os.chdir(original_cwd)\n        sys.argv = original_argv\n        sys.path[:] = original_path\n        sys.dont_write_bytecode = original_bytecode\n\n    if not checkpoint_loaded or forwards == 0:\n        raise RuntimeError("Chưa xác nhận upstream đã nạp checkpoint XD và chạy forward.")\n\n    latency_summary = latency.save(\n        output_dir, "xd", VARIANT, RUN_MODE, torch.device("cuda:0"),\n    )\n    result = {\n        "parameters": parameter_counts,\n        "latency": latency_summary,\n        **receipt,\n        "run_kind": "smoke_inference" if RUN_MODE == "smoke" else "full_checkpoint_inference",\n        "runtime_patch": "CLIP imports packaging.version directly",\n        "clip_backbone_sha256": CLIP_SHA256,\n        "forward_calls": forwards,\n        "visual_block_length": defaults.visual_length,\n        "upstream_completed": upstream_completed,\n        "post_ap_error": post_ap_error,\n        "gpu": torch.cuda.get_device_name(0),\n        "torch": torch.__version__,\n        "elapsed_seconds": elapsed,\n        "peak_torch_vram_mib": torch.cuda.max_memory_allocated() / 2**20,\n        "metrics": None,\n    }\n    if full_run:\n        if len(ap_calls) != 2:\n            raise RuntimeError(\n                f"Cần hai AP cấp frame từ xd_test.py, nhận {len(ap_calls)}. "\n                f"Xem {output_dir / \'upstream.log\'}."\n            )\n        scores = np.column_stack([column for _, column in ap_calls])\n        result["paper_table_1_frame_ap"] = PAPER_AP\n        result["metrics"] = {\n            f"branch_{index + 1}": {\n                "frame_ap": ap,\n                "roc_auc": float(metrics.roc_auc_score(gt, np.repeat(column, FRAMES_PER_FEATURE))),\n                "gap_vs_paper_percentage_points": 100 * (ap - PAPER_AP),\n            }\n            for index, (ap, column) in enumerate(ap_calls)\n        }\n        score_columns = ["branch_1", "branch_2"]\n        steps_per_video = [item["steps"] for item in inventory]\n        result["score_columns"] = score_columns\n        np.savez_compressed(\n            output_dir / "predictions.npz",\n            official_path=np.asarray([item["official_path"] for item in inventory]),\n            video_label=np.asarray([item["label"] for item in inventory]),\n            snippet_offsets=np.cumsum([0] + steps_per_video),\n            score_columns=np.asarray(score_columns),\n            scores=scores,\n            frame_gt=gt,\n        )\n    elif forwards != smoke_forwards:\n        raise RuntimeError("Smoke test chưa đủ lượt forward.")\n\n    (output_dir / "run.json").write_text(\n        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    if full_run:\n        if post_ap_error is not None:\n            print(f"Script gốc lỗi sau khi tính đủ hai AP. Xem upstream.log: {post_ap_error}")\n        for name, values in result["metrics"].items():\n            if name == "branch_2":\n                print(\n                    f"{name} [XD primary / alignment]: AP {values[\'frame_ap\'] * 100:.4f}% "\n                    f"(vs paper 84.51%: {values[\'gap_vs_paper_percentage_points\']:+.4f} pp; "\n                    "not a delta vs a previous V0 run)"\n                )\n            else:\n                print(f"{name} [auxiliary / classification]: AP {values[\'frame_ap\'] * 100:.4f}%")\n    else:\n        print("Smoke đạt, gồm video dài hơn block.")\n    print(f"Đầu ra: {output_dir}")\n\n\ntry:\n    if RUN_MODE == "preflight":\n        print("Đã kiểm input. Chọn GPU và RUN_MODE = \'smoke\' để thử checkpoint.")\n    else:\n        run_upstream()\nfinally:\n    stage_directory.cleanup()\n'


## Chạy và so sánh với V0


In [ ]:
import csv
import json
import math
import os
import subprocess
import sys
import tempfile


def runner_header(config):
    source = ["from pathlib import Path"]
    for key, value in config.items():
        code = f"Path({str(value)!r})" if isinstance(value, Path) else repr(value)
        source.append(f"{key} = {code}")
    return "\n".join(source)


def run_dataset(name, runner, config, work_dir):
    import importlib.util

    report_spec = importlib.util.spec_from_file_location(
        "benchmark_reporting", work_dir / "benchmark_reporting.py")
    reporting = importlib.util.module_from_spec(report_spec)
    report_spec.loader.exec_module(reporting)
    reporting.save_run_config(work_dir / f"{name}.config.json", config, runner)
    script = work_dir / f"{name}.py"
    script.write_text(
        runner_header(config) + "\n\n" + runner,
        encoding="utf-8",
    )
    console_path = work_dir.parent / f"{name}_console.log"
    print(f"\n--- {name} ---", flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with (
        console_path.open("w", encoding="utf-8") as console,
        subprocess.Popen(
            [sys.executable, "-u", str(script)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            encoding="utf-8", errors="replace", env=env,
        ) as process,
    ):
        for line in process.stdout:
            console.write(line)
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"Xem log lỗi: {console_path}")
    for saved in sorted(Path(config["OUTPUT"]).rglob("run.json")):
        reporting.emit_report(saved)
    return console_path

V0_REFERENCE = {
    "ucf": {"roc_auc": 0.8801992605166816, "ano_auc": 0.7023225660564671, "ap": 0.3355725526808031},
    "xd": {"frame_ap": 0.8450448809004643, "roc_auc": 0.946812650099563},
}
V0_TOLERANCE = 0.00005


def compare(prefix, paths):
    import numpy as np
    from sklearn.metrics import average_precision_score, roc_auc_score

    branch = "classification" if prefix == "ucf" else "branch_2"
    reports = [json.loads(path.read_text(encoding="utf-8")) for path in paths]
    v0, v7 = reports
    count = 290 if prefix == "ucf" else 800
    for report in reports:
        if report["run_kind"] != "full_checkpoint_inference" or report["video_count"] != count:
            raise ValueError("Báo cáo full thiếu video.")
    keys = ("split_sha256", "ground_truth_sha256", "evaluated_checkpoint_sha256",
            "visual_block_length", "frames_per_feature")
    if any(v0[key] != v7[key] for key in keys):
        raise ValueError("V0/V7 lệch split, checkpoint hoặc ánh xạ frame.")
    rows = []
    for metric, ref in V0_REFERENCE[prefix].items():
        baseline, changed = (report["metrics"][branch][metric] for report in reports)
        if not all(math.isfinite(score) and 0 <= score <= 1 for score in (baseline, changed)):
            raise ValueError(f"Metric không hợp lệ: {prefix}/{metric}.")
        if abs(baseline - ref) > V0_TOLERANCE:
            raise ValueError(f"V0 không khớp mốc tái lập: {prefix}/{metric}: {baseline} vs {ref}.")
        rows.append({
            "dataset": prefix, "metric": metric, "v0_pct": 100 * baseline,
            "v7_pct": 100 * changed, "v7_minus_v0_pp": 100 * (changed - baseline),
        })

    width = v0["visual_block_length"]
    frames = v0["frames_per_feature"]
    id_key = "video_id" if prefix == "ucf" else "official_path"
    with (
        np.load(paths[0].with_name("predictions.npz"), allow_pickle=False) as old,
        np.load(paths[1].with_name("predictions.npz"), allow_pickle=False) as new,
    ):
        if any(not np.array_equal(old[key], new[key]) for key in (
            id_key, "snippet_offsets", "frame_gt", "score_columns",
        )):
            raise ValueError("Dự đoán V0/V7 lệch ID, offset, ground truth hoặc thứ tự nhánh.")
        offsets = old["snippet_offsets"]
        gt = old["frame_gt"].reshape(-1)
        column = old["score_columns"].tolist().index(branch)
        a = np.repeat(old["scores"][:, column], frames)
        b = np.repeat(new["scores"][:, column], frames)

    if len(a) != len(b) or len(gt) != len(a) or offsets[-1] * frames != len(a):
        raise ValueError("Số frame dự đoán không khớp.")
    near = np.zeros(offsets[-1], dtype=bool)
    short_delta = []
    for first, end in zip(offsets[:-1], offsets[1:], strict=True):
        for boundary in range(first + width, end, width):
            near[max(first, boundary - 8):min(end, boundary + 8)] = True
        first_frame, end_frame = first * frames, end * frames
        segment = gt[first_frame:end_frame].astype(bool)
        edges = np.flatnonzero(np.diff(np.r_[False, segment, False])) + first_frame
        for start, stop in edges.reshape(-1, 2):
            if stop - start <= 48:
                short_delta.append(float(b[start:stop].max() - a[start:stop].max()))
    near = np.repeat(near, frames)
    metric_fn = roc_auc_score if prefix == "ucf" else average_precision_score
    region_metrics = {}
    for label, select in (("near_internal_boundary", near), ("outside_boundary_region", ~near)):
        target = gt[select]
        scores = [float(metric_fn(target, x[select])) for x in (a, b)] if np.unique(target).size == 2 else [None, None]
        region_metrics[label] = {
            "frames": len(target), "positives": int(target.sum()), "v0": scores[0], "v7": scores[1],
        }
    return rows, {
        "region_metrics": region_metrics,
        "short_event_count": len(short_delta),
        "short_event_peak_lower_count": sum(delta < 0 for delta in short_delta),
        "short_event_mean_peak_delta": float(np.mean(short_delta)) if short_delta else None,
    }


def run_both():
    import importlib.util
    if RUN_MODE not in {"smoke", "full"}:
        raise ValueError("RUN_MODE cần smoke hoặc full.")
    if not DATASETS or len(set(DATASETS)) != len(DATASETS) or set(DATASETS) - {"ucf", "xd"}:
        raise ValueError("DATASETS cần ucf, xd hoặc cả hai, không lặp.")
    if not INPUT.is_dir():
        raise FileNotFoundError("Gắn bốn Dataset Kaggle qua Add Input.")
    root = WORK / "vadclip_v7"
    root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run_", dir=root))
    scripts = output / "runtime_scripts"
    scripts.mkdir()
    (scripts / "v7_windows.py").write_text(WINDOW_SOURCE, encoding="utf-8")
    (scripts / "latency_inference.py").write_text(LATENCY_SOURCE, encoding="utf-8")
    (scripts / "benchmark_reporting.py").write_text(REPORT_SOURCE, encoding="utf-8")
    print("Đầu ra chung:", output, flush=True)
    shared = {
        "INPUT": INPUT, "WORK": WORK, "CLIP_WEIGHTS": CLIP_WEIGHTS,
        "LATENCY_WARMUP": LATENCY_WARMUP, "LATENCY_REPEATS": LATENCY_REPEATS,
        "SRC_REPO": Path(tempfile.gettempdir()) / "vadclip_xd_c41067f" / "upstream",
    }
    jobs = {
        "ucf": (UCF_RUNNER, {"DATA_ROOT": UCF_DATA_ROOT, "CHECKPOINT": UCF_CHECKPOINT}),
        "xd": (XD_RUNNER, {
            "DATA_ROOT": XD_DATA_ROOT, "CHECKPOINT": XD_CHECKPOINT, "FEATURE_ZIP": XD_FEATURE_ZIP,
            "REFERENCE_MANIFEST": None, "REQUIRE_PACK_MANIFEST": True, "RUN_MODE": RUN_MODE,
        }),
    }
    runs = {}
    rows = []
    for prefix in DATASETS:
        runner, config = jobs[prefix]
        paths = []
        entry = {"status": "running", "reports": {}}
        runs[prefix] = entry
        try:
            for variant in ("v0", "v7"):
                dest = output / prefix / variant
                log = run_dataset(f"{prefix}_{variant}", runner, {
                    **shared, **config, "VARIANT": variant, "OUTPUT": dest,
                    "SMOKE_ONLY": RUN_MODE == "smoke",
                }, scripts)
                base = dest / RUN_MODE
                candidates = [base / "run.json"] if prefix == "ucf" else list(base.glob("run_*/run.json"))
                if len(candidates) != 1 or not candidates[0].is_file():
                    raise ValueError(f"Thiếu báo cáo {prefix}/{variant}.")
                paths.append(candidates[0])
                entry["reports"][variant] = {"run_json": str(candidates[0]), "log": str(log)}
            if RUN_MODE == "full":
                metrics, entry["diagnostics"] = compare(prefix, paths)
                rows.extend(metrics)
            entry["status"] = "ok"
        except (OSError, RuntimeError, KeyError, TypeError, ValueError) as exc:
            entry.update(status="failed", error=str(exc))
            print(f"{prefix} lỗi: {exc}", flush=True)

    latency_spec = importlib.util.spec_from_file_location(
        "latency_results", scripts / "latency_inference.py")
    latency_module = importlib.util.module_from_spec(latency_spec)
    latency_spec.loader.exec_module(latency_module)
    latency_module.collect_latency(output)
    report_spec = importlib.util.spec_from_file_location(
        "benchmark_reporting", scripts / "benchmark_reporting.py")
    reporting = importlib.util.module_from_spec(report_spec)
    report_spec.loader.exec_module(reporting)
    reporting.collect_reports(output)
    summary = output / "summary.json"
    summary.write_text(json.dumps({"mode": RUN_MODE, "runs": runs, "metrics": rows},
                                  indent=2, ensure_ascii=False), encoding="utf-8")
    if rows:
        with (output / "comparison.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        for row in rows:
            print(f"{row['dataset']} {row['metric']}: {row['v0_pct']:.4f}% → "
                  f"{row['v7_pct']:.4f}% ({row['v7_minus_v0_pp']:+.4f} điểm phần trăm)")
    from zipfile import ZIP_DEFLATED, ZipFile
    archive = output / "latency_results.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED) as zipped:
        for item in sorted(output.rglob("*")):
            if item.is_file() and item.suffix in {".csv", ".json", ".log", ".py"}:
                zipped.write(item, item.relative_to(output))
    print("Download:", archive)
    print("Đầu ra chung:", output)
    if any(entry["status"] != "ok" for entry in runs.values()):
        raise RuntimeError(f"Có lượt chưa hoàn tất. Xem {summary}.")

    return archive


RESULT_ZIP = run_both()


In [ ]:
# Run only this cell after inference to prepare a relative download link.
import shutil
from pathlib import Path

from IPython.display import FileLink, display

source_zip = Path(RESULT_ZIP).resolve()
if not source_zip.is_file():
    raise FileNotFoundError(f"Results archive missing: {source_zip}")
download_zip = Path.cwd() / f"v7_{source_zip.parent.name}_results.zip"
if source_zip != download_zip.resolve():
    shutil.copy2(source_zip, download_zip)
display(FileLink(download_zip.name))
print("If the link fails, download this ZIP in the Kaggle Output/Files panel:", download_zip)
